# Question 4: Implement ReAct Agent with Multiple Tools (25 points)

Implement a ReAct (Reasoning and Acting) agent as described by Yao et al. [1], incorporating three main tools: search, compare, and analyze. This agent should be able to handle complex queries by reasoning about which tool to use and when.

a) (4 points) Implement the search tool using the SerpAPI integration from previous questions. Ensure it can be easily used by the ReAct agent.
   - Proper integration with SerpAPI
   - Formatting the search results for use by the ReAct agent

b) (5 points) Create a custom comparison tool using LangChain's `Tool` class. The tool should accept multiple items and a category as input and return a comparison result.
   - Implementing the comparison logic
   - Creating an appropriate prompt template for the comparison
   - Proper error handling for invalid inputs

c) (5 points) Implement an analysis tool that can summarize and extract key information from search results or comparisons. This tool should use the OpenAI model to generate insightful analyses.
   - Implementing the analysis logic
   - Creating an appropriate prompt template for the analysis
   - Ensuring the analysis output is concise and relevant

d) (6 points) Integrate these tools with a ReAct agent using LangChain. Your implementation should:
   - Use LangChain's `initialize_agent` function with the `AgentType.ZERO_SHOT_REACT_DESCRIPTION` agent type
   - Include all three tools (search, compare, analyze) as available actions for the agent
   - Implement proper error handling and fallback strategies
   - Ensure smooth transitions between tools in the agent's reasoning process

e) (5 points) Implement a simple Streamlit user interface for your ReAct agent. Your implementation should include:
   - A text input field for users to enter their queries
   - A button to submit the query and trigger the ReAct agent
   - A display area for showing the final results
   - A section to display the step-by-step reasoning process of the ReAct agent

In [ ]:
# Install required packages
# Pinned versions verified to work with this notebook
!pip install \
    "langchain==0.3.25" \
    "langchain-community==0.3.24" \
    "langchain-core==0.3.62" \
    "langchain-openai==0.2.14" \
    "google-search-results==2.4.2" \
    "streamlit>=1.32.0"

In [ ]:
# Import necessary libraries
import os
from google.colab import userdata
from langchain_openai import OpenAI
from langchain_core.tools import Tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents import initialize_agent, AgentType
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

# Set API keys
os.environ["OPENAI_API_KEY"]  = userdata.get("OPENAI_API_KEY")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")

# Initialize the OpenAI language model
llm = OpenAI(temperature=0)


## a) Implement the search tool

In [ ]:
# Load the search tool using SerpAPI
search_tool = load_tools(["serpapi"], llm=llm)[0]
print(f"Search tool loaded: {search_tool.name}")


Search tool loaded: Search


## b) Create a custom comparison tool

In [ ]:
def compare_items(query: str) -> str:
    """Compare multiple items in a given category.

    The tool expects `query` to follow the format:
        "item1, item2, ..., category"
    For example: "iPhone 15 Pro, Samsung Galaxy S24 Ultra, smartphones"

    Returns a string summarising the comparison, or an error message
    if the input cannot be parsed.
    """
    try:
        # Split on commas; last token is category, rest are items
        parts = [p.strip() for p in query.split(",")]
        if len(parts) < 3:
            return "Error: please provide at least two items and a category."
        items = parts[:-1]
        category = parts[-1]

        # Prompt template for comparison
        comparison_template = """Compare the following {category} in detail.
Items to compare: {items}

For each item, discuss the key features, strengths, and weaknesses.
Then provide a summary table and an overall recommendation.

Comparison:"""

        comparison_prompt = PromptTemplate(
            input_variables=["items", "category"],
            template=comparison_template
        )

        # Run the chain
        comparison_chain = LLMChain(llm=llm, prompt=comparison_prompt)
        result = comparison_chain.invoke({
            "items": ", ".join(items),
            "category": category
        })["text"]
        return result.strip()

    except Exception as e:
        return f"Error in compare_items: {str(e)}"


# Wrap in a LangChain Tool
compare_tool = Tool(
    name="Compare",
    func=compare_items,
    description=(
        "Useful for comparing multiple items within a category. "
        "Input should be a comma-separated list where the last item is the category. "
        "Example input: 'iPhone 15 Pro, Samsung Galaxy S24 Ultra, smartphones'"
    )
)
print(f"Compare tool created: {compare_tool.name}")


Compare tool created: Compare


## c) Implement an analysis tool

In [ ]:
def analyze_results(query: str) -> str:
    """Summarize and extract key information from text passed by the agent.

    Note: LangChain Tool functions must accept a *single* string argument.
    The agent will pass whatever text it wants analyzed directly as `query`.

    Returns a concise, insightful analysis string.
    """
    analysis_template = """Analyze the following text and provide a concise summary.
Highlight the key takeaways, important facts, and any notable patterns or insights.
Keep the analysis focused and actionable.

Text to analyze:
{text}

Concise Analysis:"""

    analysis_prompt = PromptTemplate(
        input_variables=["text"],
        template=analysis_template
    )

    analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)
    result = analysis_chain.invoke({"text": query})["text"]
    return result.strip()


# Wrap in a LangChain Tool
analyze_tool = Tool(
    name="Analyze",
    func=analyze_results,
    description=(
        "Useful for summarizing and extracting key insights from text. "
        "Input should be the text you want analyzed. "
        "Returns a concise summary with key takeaways."
    )
)
print(f"Analyze tool created: {analyze_tool.name}")


Analyze tool created: Analyze


## d) Integrate tools with a ReAct agent

In [ ]:
# Build the list of tools for the ReAct agent
tools = [search_tool, compare_tool, analyze_tool]

# Initialize the ReAct agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True,
)
print("ReAct agent initialized with tools:", [t.name for t in tools])


ReAct agent initialized with tools: ['Search', 'Compare', 'Analyze']


/tmp/ipykernel_1210/2141966855.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [ ]:
def process_query(query: str, max_steps: int = 100) -> str:
    """Run the ReAct agent on `query` and return its final answer.

    Args:
        query: The natural-language question to answer.
        max_steps: Maximum number of reasoning steps before stopping.

    Returns:
        The agent's final answer as a string, or an error message.
    """
    try:
        output = agent.invoke({"input": query})
        return output["output"]
    except Exception as e:
        return f"Error processing query: {str(e)}"


## e) Streamlit User Interface

Create a separate Python file (e.g. `app.py`) containing a Streamlit app for your ReAct agent. Your app must include:

- A **text input** field for the user's query
- A **submit button** to trigger the agent
- A **results area** that displays the final answer
- *(Optional)* A collapsible section showing the step-by-step Thought / Action / Observation trace

> **Note:** Streamlit apps must be run from the terminal with `streamlit run app.py`. You can use `%%writefile app.py` in a code cell to write the file directly from this notebook.

In [ ]:
%%writefile app.py
import os
import streamlit as st
from langchain_openai import OpenAI
from langchain_core.tools import Tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents import initialize_agent, AgentType
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

# ── API Keys ─────────────────────────────────────────────────────────────────
# When running via `streamlit run app.py`, read keys from environment or
# Streamlit secrets (.streamlit/secrets.toml).
os.environ.setdefault("OPENAI_API_KEY",  st.secrets.get("OPENAI_API_KEY", ""))
os.environ.setdefault("SERPAPI_API_KEY", st.secrets.get("SERPAPI_API_KEY", ""))

# ── LLM ──────────────────────────────────────────────────────────────────────
llm = OpenAI(temperature=0)

# ── Tools ────────────────────────────────────────────────────────────────────
search_tool = load_tools(["serpapi"], llm=llm)[0]

def compare_items(query: str) -> str:
    try:
        parts = [p.strip() for p in query.split(",")]
        if len(parts) < 3:
            return "Error: please provide at least two items and a category."
        items, category = parts[:-1], parts[-1]
        template = """Compare the following {category} in detail.
Items: {items}
Provide strengths, weaknesses, and a recommendation.
Comparison:"""
        chain = LLMChain(llm=llm, prompt=PromptTemplate(
            input_variables=["items", "category"], template=template))
        return chain.invoke({"items": ", ".join(items), "category": category})["text"].strip()
    except Exception as e:
        return f"Error: {e}"

compare_tool = Tool(
    name="Compare",
    func=compare_items,
    description="Compare items in a category. Input: 'item1, item2, category'",
)

def analyze_results(query: str) -> str:
    template = """Analyze the following text. Highlight key takeaways and insights.
Text: {text}
Concise Analysis:"""
    chain = LLMChain(llm=llm, prompt=PromptTemplate(
        input_variables=["text"], template=template))
    return chain.invoke({"text": query})["text"].strip()

analyze_tool = Tool(
    name="Analyze",
    func=analyze_results,
    description="Summarize and extract insights from text.",
)

# ── Agent ────────────────────────────────────────────────────────────────────
agent = initialize_agent(
    tools=[search_tool, compare_tool, analyze_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True,
    return_intermediate_steps=True,
)

# ── Streamlit UI ─────────────────────────────────────────────────────────────
st.title("ReAct Agent")
st.write("Ask a complex question and let the agent reason through it step by step.")

query = st.text_input("Enter your query:", placeholder="e.g. Compare the top 3 smartphones of 2024")

if st.button("Submit"):
    if query:
        with st.spinner("Thinking..."):
            try:
                output = agent.invoke({"input": query})
                result = output["output"]
                steps = output.get("intermediate_steps", [])
            except Exception as e:
                result = f"Error: {e}"
                steps = []

        # Display final answer
        st.subheader("Answer")
        st.write(result)

        # Display reasoning trace
        if steps:
            with st.expander("Step-by-step reasoning", expanded=False):
                for idx, (action, observation) in enumerate(steps, 1):
                    st.markdown(f"**Step {idx}**")
                    st.markdown(f"- **Thought/Action:** {action.tool} [{action.tool_input}]")
                    st.markdown(f"- **Observation:** {observation[:500]}")
                    st.divider()
    else:
        st.warning("Please enter a query before submitting.")


Writing app.py


## Test Your Implementation

Use the cell below to test your implementation with a sample query.

In [ ]:
# Test your implementation
sample_query = "What are the top 3 smartphones in 2023, and how do they compare in terms of camera quality and battery life?"

result = process_query(sample_query)
print(result)



> Entering new AgentExecutor chain...
 I should use the Compare tool to compare the top 3 smartphones in 2023.
Action: Compare
Action Input: 'iPhone 15 Pro, Samsung Galaxy S24 Ultra, Google Pixel 7, smartphones'

/tmp/ipykernel_1210/1796893263.py:34: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  comparison_chain = LLMChain(llm=llm, prompt=comparison_prompt)



Observation: iPhone 15 Pro vs Samsung Galaxy S24 Ultra vs Google Pixel 7

Key Features:

iPhone 15 Pro:
- 6.7-inch Super Retina XDR OLED display
- A15 Bionic chip with Neural Engine
- Triple-camera system with 12MP Ultra Wide, Wide, and Telephoto cameras
- LiDAR scanner for improved AR experiences
- 5G connectivity
- Face ID for secure authentication
- iOS 15 operating system
- Up to 1TB storage capacity
- Water and dust resistant (IP68)
- MagSafe for wireless charging and accessories

Samsung Galaxy S24 Ultra:
- 6.8-inch Dynamic AMOLED 2X display
- Exynos 2200 or Snapdragon 895 processor
- Quad-camera system with 108MP main, 12MP ultra-wide, 12MP telephoto, and 3D ToF sensors
- 40MP front-facing camera
- 5G connectivity
- In-display fingerprint sensor
- Android 12 operating system
- Up to 1TB storage capacity
- Water and dust resistant (IP68)
- Wireless charging and reverse wireless charging
- S Pen support

Google Pixel 7:
- 6.2-inch OLED
Thought: I should also use the Search tool t

## Submission Requirements

Please submit the following items as part of your solution:

1. Your complete code implementation for the ReAct agent and its tools (this notebook).
2. Your `app.py` Streamlit file (Part e).
3. A sample question that you used to test your tool (make it complex enough to demonstrate the use of multiple tools).
4. The final answer provided by your ReAct agent for the sample question.
5. The complete history traces of the ReAct agent for your sample question, showing its thought process, actions, and observations. Your traces should follow a format similar to this example:

```
Thought: I need to find information about top smartphones first
Action: Search[top smartphones 2023]
Observation: [Search results about top smartphones]
Thought: Now I should compare the top two options
Action: Compare[iPhone 14 Pro, Samsung Galaxy S23 Ultra, smartphones]
Observation: [Comparison result]
Thought: I should analyze this comparison for the user
Action: Analyze[comparison result]
Observation: [Analysis of the comparison]
Final Answer: [Your agent's final response to the user's query]
```

Ensure that your submission clearly demonstrates the agent's ability to reason about which tool to use and how to interpret the results from each tool. Your history traces should show a logical flow of thoughts, actions, and observations, culminating in a final answer that addresses the initial query.

**Note:** Ensure that your ReAct agent can seamlessly switch between these tools based on the task at hand. The agent should be able to reason about which tool to use next and how to interpret the results from each tool.

## References

[1] Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022). ReAct: Synergizing reasoning and acting in language models. arXiv preprint arXiv:2210.03629. https://arxiv.org/pdf/2210.03629